# ACF der Volleyball-Daten — die vier Einzelkombinationen

Ausgewertet werden die vier Kombinationen aus Geschlecht (Maenner, Frauen) und Spielniveau
(Amateur, Profi), jede fuer sich. Das Niveau *Semi Profi* wird beim Laden aus `long_df`
entfernt (Setup-Zelle, `LEVEL_ORDER`), sodass alle folgenden Zellen, Tabellen und der
Robustheitscheck automatisch auf denselben vier Gruppen arbeiten.

Die Rechenlogik liegt unveraendert im Modul `acf_analysis` (`src/acf_analysis/acf.py`) und
ist identisch zu der, die auf den simulierten Daten laeuft.

**`tau` wird frei gefittet, ohne Vergleichsfit `tau = 7`** — der Wert 7 stammt aus dem
Fussball und ist bei 14-22 Volleyballpartien nicht uebertragbar. Die Parametergrenzen
`tau in [0,5 ; 60]` bleiben.

## Setup

Import-Bootstrap ueber die Projektwurzel (Ordner mit `pyproject.toml`), damit das Notebook
mit **oder** ohne `pip install -e .` laeuft. Die Achsenbeschriftungen werden `plot_acf`
explizit uebergeben — die Modul-Defaults beschreiben den Fussballfall (`Delta m` in
Spieltagen), hier zaehlt `Delta m` Partien und `X` ist eine Punktedifferenz.

`LEVEL_ORDER` legt fest, welche Spielniveaus ausgewertet werden (hier: Amateur und Profi)
und in welcher Reihenfolge sie als Spalten im Gitter stehen. Der Filter ist die einzige
Einschraenkung des Datensatzes in diesem Notebook — **es werden keine Saisons
ausgeschlossen**. Zelle 7 am Ende zeigt, wie
viele Saisons je Geschlecht und Level tatsaechlich eingehen.

Eine Schema-/Invariantenpruefung findet hier nicht statt — der Datensatz wurde vorab auf
Felder und Qualitaet geprueft. `compute_acf` verlaesst sich allerdings auf die
Zeilensortierung (die Funktion liest `match_number` nicht, sondern nutzt die Reihenfolge der
Zeilen innerhalb jeder Team-Saison); die Aufbereitung muss `long_df` also chronologisch
sortiert liefern.

In [ ]:
import sys                                             # fuer sys.path (Import-Bootstrap)
from pathlib import Path                                 # Pfade
import numpy as np                                       # numerische Arrays
import pandas as pd                                      # Datentabellen
import matplotlib.pyplot as plt                          # Plots
from matplotlib.ticker import MaxNLocator                 # ganzzahlige Lag-Ticks

# --- acf_analysis reproduzierbar importierbar machen ----------------------
ROOT = Path.cwd()                                        # Startpunkt: Verzeichnis des Notebooks
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent                                   # nach oben, bis pyproject.toml gefunden
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from acf_analysis import compute_acf, fit_acf, bin_acf, plot_acf, acf_model  # zentrale ACF-Logik

# --- Datenquelle ---------------------------------------------------------
# data/acf-ready/ ist per .gitignore ausgeschlossen: echte Spieldaten sind sensibel
# (CLAUDE.md Abschnitt 11) und duerfen nicht ins Repo.
DATA_DIR  = ROOT / "data" / "acf-ready"
DATA_FILE = "long_df_vb.csv"              # <<< HIER den Dateinamen eintragen
SEP       = ";"

# --- Achsenbeschriftungen (Volleyball: Partien statt Spieltage) ----------
XLABEL      = r"Lag $\Delta m$ (Partien)"
YLABEL      = r"$K(\Delta m)$"
POINT_LABEL = "ACF (gebinnt, nur Darstellung)"

# --- Auswahl, Reihenfolge und Anzeigenamen der Auspraegungen -------------
# LEVEL_ORDER legt BEIDES fest: WELCHE Level ausgewertet werden und in welcher Reihenfolge
# sie als Spalten im Gitter stehen. "Semi Profi" fehlt bewusst -- ausgewertet werden nur
# Amateur und Profi, je Geschlecht -> 2x2-Gitter.
SEX_ORDER   = ['m', 'w']
LEVEL_ORDER = ['Amateur', 'Profi']
SEX_LABEL   = {"m": "Männer", "w": "Frauen"}
LEVEL_LABEL = {"Amateur": "Amateur", "Profi": "Profi"}


def lbl(value, mapping):
    """Anzeigename einer Auspraegung; faellt auf den Rohwert zurueck."""
    return mapping.get(value, str(value))


long_df = pd.read_csv(DATA_DIR / DATA_FILE, sep=SEP)

# --- auf die auszuwertenden Level einschraenken --------------------------
# EINZIGE Einschraenkung des Datensatzes in diesem Notebook. Jede season_id gehoert zu genau
# einem (sex, level), der Filter entfernt also stets VOLLSTAENDIGE Saisons und zerschneidet
# keine. Unvollstaendige Saisons bleiben bewusst drin (siehe Markdown oben).
LEVELS_IN_FILE = sorted(long_df["level"].unique())        # alle Level der Datei (Diagnose)
assert set(LEVEL_ORDER) <= set(LEVELS_IN_FILE), \
    f"LEVEL_ORDER passt nicht zur Datei; vorhanden: {LEVELS_IN_FILE}"
n_rows_all = len(long_df)                                 # Zeilenzahl VOR dem Filter
long_df = long_df[long_df["level"].isin(LEVEL_ORDER)].reset_index(drop=True)  # Reihenfolge bleibt

print(f"Datensatz    : {DATA_FILE}")
print(f"Level (Datei): {LEVELS_IN_FILE}")
print(f"ausgewertet  : {LEVEL_ORDER}   "
      f"({n_rows_all - len(long_df):,} von {n_rows_all:,} Zeilen verworfen)")
print(f"Zeilen       : {len(long_df):,}   (= 2 x Partien, Heim- und Auswaertssicht)")
print(f"Partien      : {len(long_df) // 2:,}")
print(f"Saisons      : {long_df['season_id'].nunique()}")
print(f"Team-Saisons : {long_df.groupby(['season_id', 'team_id']).ngroups}")
long_df.head()

## Zelle 0 — Diagnose und Kalibrierung von `n_min`

Liefert drei Dinge:

1. **Die tatsaechlichen Auspraegungen von `sex` und `level`** — zum Fuellen von `SEX_ORDER`,
   `LEVEL_ORDER` und der Label-Dicts im Setup.
2. **Kennzahlen je Teilgruppe** — Saisons, Team-Saisons, Partien, Spiele pro Team
   (min/median/max), groesster Lag, `Var(X)`, `N(1)`.
3. **Den `N(Delta m)`-Verlauf**, aus dem `N_MIN` folgt. Erwartet wird kein glatter Verlauf,
   sondern einer mit reproduzierbaren Einbruechen; im Extremfall gilt bei einem Lag `N = 0`
   und damit `K = NaN`. Das ist unkritisch (der Fit ueberspringt NaN-Lags, im Binning tragen
   sie 0 Paare bei), die Zelle warnt aber explizit.

Der aus dem Fussball uebernommene Wert 9000 passt hier nicht: im Volleyball liegt `N(1)` je
Teilgruppe bei ~4000-6000, die greedy-Schleife wuerde die gesamte ACF auf 3-4 Punkte
zusammenfassen. Die Zelle bestimmt die Konstante deshalb ueber dasselbe Verhaeltnis
~ 0,39 * `N(1)` neu, bezogen auf das kleinste `N(1)`, und schlaegt sie vor.

In [ ]:
print("Werte in 'sex'  :", sorted(long_df["sex"].unique()))
print("Werte in 'level':", sorted(long_df["level"].unique()))
print()

diag_rows, diag_acf = [], {}
for (s, lv), g in long_df.groupby(["sex", "level"], sort=False):
    sizes = g.groupby(["season_id", "team_id"]).size()     # Spiele je Team-Saison
    acf_g = compute_acf(g)                                 # nur zur Diagnose
    diag_acf[(s, lv)] = acf_g
    diag_rows.append({
        "sex": s, "level": lv,
        "Saisons":      g["season_id"].nunique(),
        "Team-Saisons": len(sizes),
        "Partien":      len(g) // 2,
        "Spiele/Team min": int(sizes.min()),
        "Spiele/Team med": int(sizes.median()),
        "Spiele/Team max": int(sizes.max()),
        "max Lag":      int(acf_g["lag"].max()),
        "Var(X)":       round(float(g["X"].var()), 3),
        "N(1)":         int(acf_g["N"].iloc[0]),
        "0.39*N(1)":    int(round(0.39 * acf_g["N"].iloc[0])),
    })

diag = pd.DataFrame(diag_rows)
print("Kennzahlen je Teilgruppe:")
print(diag.to_string(index=False))

profile = pd.DataFrame({f"{s}/{lv}": a.set_index("lag")["N"] for (s, lv), a in diag_acf.items()})
print("\nN(Delta m) je Teilgruppe (leer = Lag existiert in dieser Gruppe nicht):")
print(profile.to_string(na_rep="-"))

zero_lags = {f"{s}/{lv}": a.loc[a["N"] == 0, "lag"].tolist() for (s, lv), a in diag_acf.items()}
zero_lags = {k: v for k, v in zero_lags.items() if v}
if zero_lags:
    print("\nHinweis: Lags ohne ein einziges gueltiges Paar (N=0) -> K(Delta m) = NaN:")
    for k, v in zero_lags.items():
        print(f"  {k}: Delta m = {v}")
    print("  Typische Ursache: gespiegelte Doppelrunde -- bei Delta m = (Teams - 1) trifft")
    print("  JEDES Team wieder auf denselben Gegner. Unkritisch fuer Fit und Binning.")

print(f"\nVorschlag fuer N_MIN: {int(round(0.39 * diag['N(1)'].min()))}")
print(f"  (0.39 x kleinstes N(1) = 0.39 x {diag['N(1)'].min()})")

## Zelle 1 — Konfiguration der vier Auswertungen

`N_MIN` gilt als **eine** absolute Schwelle fuer alle vier Auswertungen.

`D_EVAL` ist der fest vorgegebene Bezugslag, an dem die Abfallhoehe aller vier Gruppen
verglichen wird — entscheidend ist, dass es fuer alle Gruppen derselbe Lag ist. Ausgewertet
wird dort die Fit-Kurve, nicht der einzelne Messpunkt.

`tau_status` entscheidet nicht ueber die Fit-Variante (es wird immer frei gefittet), sondern
nur darueber, ob `tau` als Messwert berichtet werden darf.

In [ ]:
# --- Binning-Schwelle: EIN absoluter Wert fuer alle vier Auswertungen ----
# TODO: nach dem ersten Lauf von Zelle 0 auf den dort vorgeschlagenen Wert setzen.
N_MIN = 1616

# --- Bezugslag fuer den Vergleich ZWISCHEN den Gruppen -------------------
# FEST VORGEGEBEN, nicht aus Paarzahlen hergeleitet. Einzige Anforderung: fuer alle vier
# Gruppen derselbe Wert. Zelle 2 prueft, dass der Lag in jeder Gruppe ueberhaupt existiert.
D_EVAL = 12                                               # Bezugslag der Abfallhoehe

# --- Reihenfolge der Auspraegungen --------------------------------------
sexes  = list(SEX_ORDER)   if SEX_ORDER   else sorted(long_df["sex"].unique())
levels = list(LEVEL_ORDER) if LEVEL_ORDER else sorted(long_df["level"].unique())
assert set(sexes)  == set(long_df["sex"].unique()),   "SEX_ORDER passt nicht zu den Daten"
assert set(levels) == set(long_df["level"].unique()), "LEVEL_ORDER passt nicht zu den Daten"

# --- die vier Auswertungen; row/col legen die Position im 2x2-Gitter fest --
# Zeilen = Geschlecht (SEX_ORDER), Spalten = Level (LEVEL_ORDER, ohne Semi Profi)
SUBSETS = [{"key": f"{s}_{lv}", "label": f"{lbl(s, SEX_LABEL)} — {lbl(lv, LEVEL_LABEL)}",
            "sex": s, "level": lv, "row": i, "col": j}
           for i, s in enumerate(sexes) for j, lv in enumerate(levels)]

# --- sitzt tau auf einer Parametergrenze? -------------------------------
TAU_BOUNDS = (0.5, 60.0)                                  # muss zu den bounds in fit_acf passen


def tau_status(fit, tol=0.01, rel_err_max=0.5):
    """Ist tau ein Messwert oder eine Anzeige?

    Auf einer Grenze ist tau NICHT bestimmt: unten degeneriert das Modell zur Konstanten,
    oben zur Geraden. Auch innerhalb der Grenzen kann tau wertlos sein, naemlich wenn sein
    relativer Fehler >= rel_err_max ist. Argumentiert wird in allen drei Faellen ueber den
    Rueckgang von K, nicht ueber tau.
    """
    lo, hi = TAU_BOUNDS
    if fit.tau <= lo * (1 + tol):
        return "an Untergrenze"                           # Modell ~ konstant, Szenario (a)
    if fit.tau >= hi * (1 - tol):
        return "an Obergrenze"                            # Modell ~ linear, Szenario (d)
    if (not np.isfinite(fit.tau_err)) or fit.tau_err >= rel_err_max * abs(fit.tau):
        return "unbestimmt"                               # innerhalb der Grenzen, aber wertlos
    return "frei"                                          # Abfall auf ein Plateau, Szenario (c)


print(f"N_MIN  = {N_MIN}   (eine absolute Schwelle fuer alle {len(SUBSETS)} Auswertungen)")
print(f"D_EVAL = {D_EVAL}     (fester Bezugslag der Abfallhoehe, fuer alle Gruppen gleich)")
print(f"Geschlechter: {sexes}")
print(f"Level       : {levels}")
print(f"\n{len(SUBSETS)} Auswertungen konfiguriert (X bleibt roh, keine Standardisierung):")
for sub in SUBSETS:
    print(f"  [{sub['row']},{sub['col']}]  {sub['label']}")

## Zelle 2 — Batch: ACF, Fit und Binning

Je Auswertung: filtern → `compute_acf` → `fit_acf` → `bin_acf`. Alle drei Schritte kommen aus
dem Modul, es wird nichts neu implementiert. Boolesches Filtern erhält die Zeilenreihenfolge,
auf die sich `compute_acf` verlässt — nach dem Filtern ist kein erneutes Sortieren nötig.

Berechnet werden zusätzlich `K(1)` und `K(D_EVAL)` aus der Fit-Kurve. Der Rückgang zwischen
beiden ist die Größe, auf der die Interpretation ruht.

In [ ]:
results = {}

for sub in SUBSETS:
    df = long_df[(long_df["sex"] == sub["sex"]) & (long_df["level"] == sub["level"])]
    acf  = compute_acf(df)                                # ACF K(Delta m), N, S, SQ, sigma
    fit  = fit_acf(acf)                                   # freier Fit (tau=7 wird ignoriert)
    bins = bin_acf(acf, n_min=N_MIN)                      # Binning NUR fuer die Darstellung
    sizes = df.groupby(["season_id", "team_id"]).size()

    results[sub["key"]] = {
        "sub": sub, "label": sub["label"], "acf": acf, "fit": fit, "bins": bins,
        "tau_status":     tau_status(fit),
        "n_seasons":      int(df["season_id"].nunique()),
        "n_team_seasons": int(len(sizes)),
        "n_matches":      int(len(df) // 2),
        "games_med":      float(sizes.median()),
        "max_lag":        int(acf["lag"].max()),
        "n_pairs":        int(acf["N"].sum()),
        "var_X":          float(df["X"].var()),
    }

# --- Bezugslag pruefen: D_EVAL muss in JEDER Gruppe ein realer Lag sein ---
_zu_kurz = {r["label"]: r["max_lag"] for r in results.values() if r["max_lag"] < D_EVAL}
assert not _zu_kurz, (f"D_EVAL = {D_EVAL} liegt jenseits des groessten Lags von: {_zu_kurz} "
                      f"-- D_EVAL in Zelle 1 kleiner waehlen.")

for r in results.values():                                # Kurvenwerte an den zwei Bezugslags
    fit = r["fit"]
    r["K_1"]    = float(acf_model(1.0,              fit.a, fit.b, fit.tau))
    r["K_eval"] = float(acf_model(float(D_EVAL),    fit.a, fit.b, fit.tau))
    r["rest"]   = r["K_eval"] / r["K_1"] if abs(r["K_1"]) > 1e-12 else np.nan

print(f"Bezugslag D_EVAL = {D_EVAL} (fest vorgegeben, fuer alle Gruppen gleich)\n")
for r in results.values():
    fit = r["fit"]
    print(f"### {r['label']}   ({len(r['acf'])} Lags, {r['n_pairs']:,} Paare, "
          f"{len(r['bins'])} Bins)")
    print(f"    Fit : a={fit.a:.4f}  b={fit.b:.4f}  tau={fit.tau:.2f}+-{fit.tau_err:.2f}  "
          f"chi2/dof={fit.chi2_red:.2f}  corr(b,tau)={fit.corr_b_tau:+.2f}")
    print(f"    tau : {r['tau_status']}")
    print(f"    K(1) = {r['K_1']:.4f}  ->  K({D_EVAL}) = {r['K_eval']:.4f}"
          f"   (Restanteil {r['rest']:.3f})")
    print()

## Zelle 3 — Ergebnistabellen

**Kennzahlen** — Umfang jeder Auswertung, `Var(X)` und die Zahl der Bins bei `N_MIN`.

In [ ]:
# --- Tabelle 1: Kennzahlen ----------------------------------------------
kenn = pd.DataFrame([{
    "Auswertung":   r["label"],
    "Saisons":      r["n_seasons"],
    "Team-Saisons": r["n_team_seasons"],
    "Partien":      r["n_matches"],
    "Spiele/Team (med)": r["games_med"],
    "max Lag":      r["max_lag"],
    "Paare gesamt": r["n_pairs"],
    "Bins":         len(r["bins"]),
    "Var(X)":       round(r["var_X"], 3),
} for r in results.values()])
print(f"Kennzahlen (N_MIN = {N_MIN}):")
print(kenn.to_string(index=False))

# --- Tabelle 2: Fit-Parameter -------------------------------------------
fitp = pd.DataFrame([{
    "Auswertung":  r["label"],
    "a":           round(r["fit"].a, 4),
    "a+-":         round(r["fit"].a_err, 4),
    "b":           round(r["fit"].b, 4),
    "b+-":         round(r["fit"].b_err, 4),
    "tau":         round(r["fit"].tau, 2),
    "tau+-":       round(r["fit"].tau_err, 2),
    "tau-Status":  r["tau_status"],
    "K(1)":        round(r["K_1"], 4),
    f"K({D_EVAL})":   round(r["K_eval"], 4),
    "Restanteil":  round(r["rest"], 3),
    "chi2/dof":    round(r["fit"].chi2_red, 2),
    "a_norm":      round(r["fit"].a / r["var_X"], 4),
    "b_norm":      round(r["fit"].b / r["var_X"], 4),
    "corr(b,tau)": round(r["fit"].corr_b_tau, 2),
} for r in results.values()])
print(f"\nFit-Parameter (freier Fit; Restanteil = K({D_EVAL})/K(1)):")
print(fitp.to_string(index=False))

# --- Lesart und Rangfolge ------------------------------------------------
DEUTUNG = {
    "an Obergrenze":  "(d) Drift -- faellt ueber die ganze Saison, kein Plateau erreicht",
    "an Untergrenze": "(a) konstante Staerke -- flaches Plateau, nur ein Ein-Partie-Effekt",
    "unbestimmt":     "tau nicht aufgeloest (rel. Fehler >= 50 %) -- ueber den Rueckgang argumentieren",
    "frei":           "(c) Formphasen -- exponentieller Abfall auf ein Plateau",
}
print("\nLesart (Szenarien aus CLAUDE.md Abschnitt 5):")
for r in results.values():
    print(f"  - {r['label']:30s} tau {r['tau_status']:15s} Restanteil {r['rest']:5.3f}  "
          f"-> {DEUTUNG[r['tau_status']]}")

print(f"\nRangfolge nach Restanteil bei d = {D_EVAL} (flachste Kurve zuerst):")
for r in sorted(results.values(), key=lambda x: -x["rest"]):
    print(f"  {r['rest']:6.3f}  {r['label']}")

## Zelle 3b — Abfallhoehe bei `Delta m = D_EVAL` (normierte Modellfunktion)

Gibt je Gruppe eine Zahl aus: den Wert der Fit-Kurve bei `Delta m = D_EVAL`, normiert auf
`a + b = K_fit(0)` — dieselbe Normierung, die auch `plot_acf` verwendet. Nach der Division
startet jede Kurve bei `Delta m = 0` im Wert 1, und die vier Gruppen werden trotz sehr
unterschiedlicher `Var(X)` direkt vergleichbar.

Gerechnet wird mit der Modellfunktion statt mit dem ACF-Messwert bei `D_EVAL`: die Fit-Kurve
stammt aus allen Lags und ist an dieser Stelle stabiler als ein einzelner Lag. Die gebinnten
Messpunkte im Plot (Zelle 4) sind die Kontrolle.

Zum Vergleich steht der bisherige Restanteil `K(D_EVAL)/K(1)` daneben; er unterscheidet sich
nur durch den Nenner.

In [ ]:
# --- Abfallhoehe: Modellfunktion bei Delta m = D_EVAL, normiert auf a + b ---
# Normierung wie in plot_acf: geteilt wird durch K_fit(0) = a + b (CLAUDE.md Abschnitt 6).
# Ausgewertet wird die FIT-KURVE (aus allen Lags), nicht der einzelne Messpunkt bei D_EVAL.

abfall_rows = []
for r in results.values():
    fit  = r["fit"]
    k0   = fit.a + fit.b                                     # = K_fit(0), Achsenabschnitt
    K_d  = float(acf_model(float(D_EVAL), fit.a, fit.b, fit.tau))   # Modellwert bei D_EVAL
    khat = K_d / k0 if abs(k0) > 1e-12 else np.nan           # normiert -> Vergleichsgroesse
    n_d  = r["acf"].loc[r["acf"]["lag"] == D_EVAL, "N"]      # Paarzahl dort (nur Diagnose)
    abfall_rows.append({
        "Auswertung":              r["label"],
        "a+b":                     round(k0, 4),
        f"K({D_EVAL})":            round(K_d, 4),
        f"K({D_EVAL})/(a+b)":      round(khat, 4),           # <<< die Vergleichsgroesse
        "Abfall %":                round(100 * (1 - khat), 1),
        f"K({D_EVAL})/K(1)":       round(r["rest"], 4),      # bisheriger Restanteil, zum Vergleich
        f"N({D_EVAL})":            int(n_d.iloc[0]) if len(n_d) else 0,
    })

abfall = (pd.DataFrame(abfall_rows)
            .sort_values(f"K({D_EVAL})/(a+b)", ascending=False)     # flachste Kurve zuerst
            .reset_index(drop=True))

print(f"Abfallhoehe der Modellfunktion bei Delta m = {D_EVAL} "
      f"(normiert auf a+b = K_fit(0)):")
print(abfall.to_string(index=False))

print(f"\nRangfolge nach K({D_EVAL})/(a+b) -- flachste Kurve zuerst:")
for _, row in abfall.iterrows():
    print(f"  {row[f'K({D_EVAL})/(a+b)']:6.4f}   {row['Auswertung']:20s} "
          f"(Abfall {row['Abfall %']:5.1f} % gegenueber Delta m = 0)")

# --- wie gut ist dieser Lag in den Daten gestuetzt? ----------------------
duenn = abfall.loc[abfall[f"N({D_EVAL})"] < N_MIN, "Auswertung"].tolist()
if duenn:
    print(f"\nHinweis: bei Delta m = {D_EVAL} liegt N unter N_MIN = {N_MIN} in: "
          f"{', '.join(duenn)}.")
    print("  Unkritisch, weil hier die FIT-KURVE ausgewertet wird und der Fit auf ALLEN Lags")
    print("  laeuft -- der duenn besetzte Einzel-Lag geht nicht gesondert ein. Die Zahl steht")
    print("  nur da, damit sichtbar bleibt, wie viele Paare an dieser Stelle real vorliegen.")

## Zelle 4 — Plot: alle vier Auswertungen in einer Grafik

2x2-Gitter, Zeilen = Geschlecht (Maenner, Frauen), Spalten = Spielniveau (Amateur, Profi).
Gebinnte ACF-Punkte mit Fehlerbalken, darueber die Fit-Kurve aus allen ungebinnten Lags —
nur der freie Fit, ohne Legendenkasten (ein `tau = 60,00` in der Legende wuerde einen
Messwert suggerieren, wo `tau` auf der Parametergrenze sitzt).

Die y-Achsen sind nicht geteilt: `X` bleibt roh, `K` hat die Einheit `X^2`, und `Var(X)`
unterscheidet sich zwischen den Gruppen um mehr als Faktor 2. Fuer den Vergleich zwischen
den Panels sind der `Restanteil` in Zelle 3 und die normierte Abfallhoehe in Zelle 3b da.

Das Binning dient nur der Darstellung und geht nicht in den Fit ein.

In [ ]:
# Gitterform folgt der Konfiguration: 2 Geschlechter x 2 Level -> 2x2
fig, axes = plt.subplots(len(sexes), len(levels),
                         figsize=(5.4 * len(levels), 4.6 * len(sexes)))
axes = np.atleast_2d(axes)

for sub in SUBSETS:
    r  = results[sub["key"]]
    ax = axes[sub["row"], sub["col"]]
    plot_acf(r["bins"], r["fit"], primary="free", ax=ax, title=r["label"],
             point_label=POINT_LABEL,
             xlabel=XLABEL if sub["row"] == len(sexes) - 1 else None,   # nur untere Reihe
             ylabel=YLABEL if sub["col"] == 0 else None)                # nur linke Spalte
    # Delta m zaehlt Partien und ist ganzzahlig; bei den kurzen Volleyball-Lagspannen
    # (Delta m ~ 1..12) waehlt Matplotlibs Auto-Locator sonst Schrittweiten von 0.5.
    # Betrifft nur die Ticks -- die gebinnten Punkte behalten ihre paarzahlgewichtete
    # x-Position aus bin_acf.
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

fig.suptitle("ACF der bereinigten Punktedifferenz — Volleyball, nach Geschlecht und Level",
             fontsize=14)
fig.tight_layout()
plt.show()

## Zelle 5 — Export

Geschrieben werden:

- `results/real_acf_vb_simple_{key}.csv` — ACF-Tabelle je Auswertung (`lag`, `K`, `N`, `sigma`)
- `results/real_acf_vb_simple_kennzahlen.csv` und `..._fitparameter.csv`
- `figures/acf_vb_simple.png` — die Abbildung fuer die Bachelorarbeit

Exportiert werden ausschliesslich aggregierte Groessen (`K(Delta m)`, Paarzahlen,
Fit-Parameter) — keine Einzelergebnisse, keine Vereinsnamen. `SAVE = False` setzen, wenn
nichts geschrieben werden soll.

In [ ]:
SAVE = True

RESULTS_DIR = ROOT / "results"
FIGURES_DIR = ROOT / "figures"

if SAVE:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

    for key, r in results.items():
        path = RESULTS_DIR / f"real_acf_vb_simple_{key}.csv"
        r["acf"][["lag", "K", "N", "sigma"]].to_csv(path, index=False)
        print(f"geschrieben: {path.relative_to(ROOT)}")

    kenn.to_csv(RESULTS_DIR / "real_acf_vb_simple_kennzahlen.csv", index=False)
    fitp.to_csv(RESULTS_DIR / "real_acf_vb_simple_fitparameter.csv", index=False)
    print(f"geschrieben: results/real_acf_vb_simple_kennzahlen.csv")
    print(f"geschrieben: results/real_acf_vb_simple_fitparameter.csv")

    path = FIGURES_DIR / "acf_vb_simple.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print(f"geschrieben: {path.relative_to(ROOT)}")
else:
    print("SAVE = False -- nichts geschrieben.")